Load and Format Prompt

In [15]:
PROMPT_PATH = "L_and_I_Prompt_ENACTING.txt"
with open(PROMPT_PATH, "r") as file:
    prompt_raw = file.read()

In [22]:
# Used to separate system message 
SPLIT_STRING = "\n[***NEW_MESSAGE***]\n"

# Split the raw prompt text into 3 sections
prompt_split = prompt_raw.split(SPLIT_STRING)

# Ensure the prompt file format is correct
assert len(prompt_split) == 3, "Prompt file format incorrect."

In [24]:
# Convert prompt components to Amplify message format(json)
messages = [
    {"role": "system", "content": prompt_split[0].strip()},
    {"role": "user", "content": prompt_split[1].strip()},
    {"role": "assistant", "content": prompt_split[2].strip()}
]

# (Optional)Print to verify structure
for m in messages:
    print(m)


{'role': 'system', 'content': 'You are a helpful research assistant whose job it is to identify segments of students\' self-regulated learning (SRL) behaviors using multimodal data of them interacting with a mixed-reality embodied learning environment. Specifically, your job is to identify segments where students are demonstrating the ENACTING behavior.\n\nThis work focuses on the photosynthesis model. Participants\' movements are displayed on a projector screen while students embody different molecules as they move around the classroom. The environment features a mouse and a tomato plant with zoomed-in chloroplasts and roots (areas that cause molecule transformations). Students must move towards these areas to model interactions between the molecules they are embodying (oxygen, water, water-thinking, sugar and carbon dioxide) and the features on the screen. As students move, their states change, and they learn about the process of photosynthesis.\n\nFor example, one student may be ass

Import data

In [28]:
import pandas as pd

DATA_PATH = "L&I - Student Mastersheet - EN-MESSI-D2.csv"

df = pd.read_csv(DATA_PATH)

# Convert all the NaN to the ''
df = df.fillna('')

# Convert instances of "not moving" to "stationary"
df["data"] = df["data"].replace("not moving", "stationary")

df.head()

,start_time,end_time,modality,data
0,0:00:00,,state,waterthinking
1,0:00:00,0:00:05,movement,stationary
2,0:00:00,0:00:01,gaze,Student - Taylor Swift
3,0:00:06,0:00:09,movement,moving
4,0:00:07,0:04:38,gaze,Screen


In [30]:
# Filter dataframe where:
# 1. "modality" does not include gesture or speech
# 2. "gaze" only includes Screen
df_no_speech_or_gaze = df[~df["modality"].isin(["gesture", "speech"])]
assert set(df_no_speech_or_gaze.modality) == {'state', 'movement', 'action', 'gaze'}

df_no_speech_or_gaze_screen_only = df_no_speech_or_gaze[(df_no_speech_or_gaze["modality"] != "gaze") | \
        (df_no_speech_or_gaze["data"] == "Screen")]
assert all(df_no_speech_or_gaze_screen_only[df_no_speech_or_gaze_screen_only["modality"] == \
                                            "gaze"]["data"] == "Screen")
df_no_speech_or_gaze_screen_only.head(5)

,start_time,end_time,modality,data
0,0:00:00,,state,waterthinking
1,0:00:00,0:00:05,movement,stationary
3,0:00:06,0:00:09,movement,moving
4,0:00:07,0:04:38,gaze,Screen
5,0:00:08,0:00:14,movement,stationary


In [48]:
# Make data comma-separated form but in string
data_csv_string = df_no_speech_or_gaze_screen_only.to_csv(index=False)

data_csv_string_split = [s for s in data_csv_string.split('\n') if s]
assert len(data_csv_string_split) == len(df_no_speech_or_gaze_screen_only)+1,print(len(data_csv_string_split), len(df_no_speech_or_gaze_screen_only)+1)

# Debugging
# print(len(data_csv_string_split))
# for line in data_csv_string_split:
#     print(line)

In [52]:
# Rebuild messages cleanly every time

messages = [
    {"role": "system", "content": prompt_split[0].strip()},
    {"role": "user", "content": prompt_split[1].strip()},
    {"role": "assistant", "content": prompt_split[2].strip()},
    {"role": "user", "content": data_csv_string}
]

print("Message length:", len(messages))


Message length: 4


Inference

In [ ]:
# Inference via Vanderbilt Amplify API
import requests
import json

In [ ]:
#1.Configuration
BASE_URL = "https://prod2-api.vanderbilt.ai"   
AMP_TOKEN = "YOUR_AMP_TOKEN_HERE"             

headers = {
    "Authorization": f"Bearer {AMP_TOKEN}",
    "Content-Type": "application/json"
}

In [ ]:
##2.Build request payload
payload = {
    "data": {
        "temperature": 0,          # deterministic output
        "max_tokens": 4000,
        "messages": messages,      # the message list we built earlier
        "options": {
            "skipRag": True,       # disable RAG
            "ragOnly": False,
            "model": {
                "id": "GPT 5.2"     
            }
        }
    }
}

In [ ]:
##3.Send POST request

response = requests.post(
    f"{BASE_URL}/chat",
    headers=headers,
    json=payload
)

In [ ]:
##4. Check HTTP status
if response.status_code != 200:
    print("HTTP Status Code:", response.status_code)
    print("Error response:")
    print(response.text)
    raise Exception("Amplify request failed")

In [ ]:
##5. Parse JSON response

result = response.json()

print("Raw Amplify Response:")
print(result)

In [ ]:
##6. Extract model output
if result.get("success"):
    text_response = result["data"]  
else:
    raise Exception("Amplify returned failure:", result)

In [ ]:
##7.Convert JSON string to Python dict

parsed_json = json.loads(text_response)

print("Parsed JSON:")
print(parsed_json)

Save Response

import os
import json

RESULTS_PATH = ""

# Ensure directory exists
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

# Save JSON safely
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(parsed_json, f, indent=4, ensure_ascii=False)

print("JSON successfully saved to:", RESULTS_PATH)
